In [1]:
import pandas as pd
from pathlib import Path

INPUT_DIR = Path('data/processed')
OUTPUT_DIR = Path('data/validated')
REPORT_DIR = Path('output/reports')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'cuti': INPUT_DIR / 'cuti_cleaned.xlsx',
    'tanggal': INPUT_DIR / 'tanggal_cleaned.xlsx',
}
DATE_COLUMNS = ['attendance_date', 'employee_start_date', 'employee_end_date', 'last_update']
TIME_COLUMNS = ['clock_in_time', 'clock_out_time']
TEXT_COLUMNS = ['employee_id', 'location_code', 'organization_code', 'shift_code']
REQUIRED_COLUMNS = ['employee_id', 'attendance_date']


def check_dates(df, source):
    reports = []
    for col in DATE_COLUMNS:
        if col not in df.columns:
            continue
        values = df[col]
        present = values.notna() & values.astype('string').str.strip().ne('').fillna(False)
        as_text = values.astype('string').str.strip()
        suspicious = as_text.str.startswith('1970-01-01', na=False)
        # A quality flag only, not a date transformation.
        reports.append({
            'source': source, 'column': col, 'dtype': str(values.dtype),
            'rows': len(values), 'missing_count': int((~present).sum()),
            'suspicious_1970_count': int(suspicious.sum()),
            'action': 'PRESERVED_UNCHANGED',
        })
        if suspicious.any():
            examples = as_text[suspicious].head(3).tolist()
            raise ValueError(
                f'{source}.{col}: {int(suspicious.sum())} suspicious 1970 dates. '
                f'Examples: {examples}. Check notebook 2 or raw data.'
            )
    return reports


def validate_dataset(df, source):
    df = df.copy(deep=True)
    missing_required = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
    if missing_required:
        raise ValueError(f'{source}: required columns missing: {missing_required}')
    original_dates = {col: df[col].copy(deep=True) for col in DATE_COLUMNS if col in df.columns}
    original_times = {col: df[col].copy(deep=True) for col in TIME_COLUMNS if col in df.columns}
    reports = check_dates(df, source)

    for col in TEXT_COLUMNS:
        if col in df.columns:
            # Only text columns are normalized; date and time columns are untouched.
            df[col] = df[col].astype('string').str.strip()

    for col, original in {**original_dates, **original_times}.items():
        pd.testing.assert_series_equal(df[col], original, check_names=True)

    reports.extend({
        'source': source, 'column': col, 'dtype': str(df[col].dtype),
        'rows': len(df), 'missing_count': int(df[col].isna().sum()),
        'suspicious_1970_count': None, 'action': 'PRESERVED_UNCHANGED',
    } for col in TIME_COLUMNS if col in df.columns)
    return df, reports


all_reports = []
for source, file_path in FILES.items():
    if not file_path.exists():
        raise FileNotFoundError(f'Missing input: {file_path}')

    df = pd.read_excel(file_path, dtype={'employee_id': 'string'})
    validated, reports = validate_dataset(df, source)
    all_reports.extend(reports)

    output_path = OUTPUT_DIR / f'{source}_validated.xlsx'
    validated.to_excel(output_path, index=False)
    reloaded = pd.read_excel(output_path, dtype={'employee_id': 'string'})
    if reloaded.shape != validated.shape or list(reloaded.columns) != list(validated.columns):
        raise AssertionError(f'{source}: row/column structure changed on export')
    for col in DATE_COLUMNS:
        if col in validated.columns:
            before = validated[col].astype('string').fillna('<MISSING>')
            after = reloaded[col].astype('string').fillna('<MISSING>')
            # Compare actual temporal values when both sides are datetime, otherwise exact text.
            if pd.api.types.is_datetime64_any_dtype(validated[col]) and pd.api.types.is_datetime64_any_dtype(reloaded[col]):
                matches = validated[col].eq(reloaded[col]) | (validated[col].isna() & reloaded[col].isna())
            else:
                matches = before.eq(after)
            if not bool(matches.all()):
                raise AssertionError(f'{source}.{col}: values changed during Excel round-trip')
    print(f'{source}: {len(validated):,} rows preserved -> {output_path}')

pd.DataFrame(all_reports).to_excel(REPORT_DIR / 'type_validation_report.xlsx', index=False)
print('Validation complete. Dates and times were not converted; no rows dropped.')


cuti: 7,415 rows preserved -> data\validated\cuti_validated.xlsx
tanggal: 27,774 rows preserved -> data\validated\tanggal_validated.xlsx
Validation complete. Dates and times were not converted; no rows dropped.
